In [ ]:
import os
import getpass

# Groq API key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY")

# ngrok auth token
NGROK_TOKEN = getpass.getpass("NGROK_TOKEN")

# Save ngrok token to local ngrok config
!ngrok config add-authtoken 3FTsv0KxCAvrtO1KUny9DBSlQtH_54ACSczezjrSjXdsBSpW7

GROQ_API_KEY··········
NGROK_TOKEN··········
/bin/bash: line 1: ngrok: command not found


In [ ]:
import subprocess
import time
from pyngrok import ngrok

process = subprocess.Popen(
    ["autogenstudio", "ui", "--port", "8081", "--host", "0.0.0.0"]
)

time.sleep(8)

public_url = ngrok.connect(8081)
print("AutoGen Studio URL:", public_url)

AutoGen Studio URL: NgrokTunnel: "https://audacious-level-hardly.ngrok-free.dev" -> "http://localhost:8081"


In [ ]:
import os
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient

async def run_team():
    # Fetch the Groq API key you securely entered in Cell 2
    groq_key = os.environ["GROQ_API_KEY"]

    # Define Groq model clients with the correct model_info settings
    researcher_model = OpenAIChatCompletionClient(
        model="llama-3.1-8b-instant",
        base_url="https://api.groq.com/openai/v1",
        api_key=groq_key,
        model_info={
            "vision": False,
            "function_calling": True,
            "json_output": True,
            "structured_output": True,
            "family": "unknown"
        }
    )

    editor_model = OpenAIChatCompletionClient(
        model="llama-3.3-70b-versatile",
        base_url="https://api.groq.com/openai/v1",
        api_key=groq_key,
        model_info={
            "vision": False,
            "function_calling": True,
            "json_output": True,
            "structured_output": True,
            "family": "unknown"
        }
    )

    # Define the individual Agents
    researcher = AssistantAgent(
        name="Researcher",
        model_client=researcher_model,
        system_message="You are an expert researcher. Provide a highly detailed summary using clear Markdown formatting."
    )

    editor = AssistantAgent(
        name="Editor",
        model_client=editor_model,
        system_message="You are a strict editor. Critique the researcher's work and optimize it for professional delivery."
    )

    # Orchestrate the workflow team
    team = RoundRobinGroupChat(
        participants=[researcher, editor],
        termination_condition=MaxMessageTermination(max_messages=4)
    )

    # Run the prompt
    print("--- Starting Multi-Agent Session ---")
    async for message in team.run_stream(
        task="Explain why Groq LPUs provide higher throughput for LLMs than standard GPUs."
    ):
        if hasattr(message, "source") and hasattr(message, "content"):
            print(f"\n\033[1m[{message.source}]\033[0m: {message.content}")
            print("-" * 40)
        else:
            print("\n--- Final Result ---")
            print(message)
            print("-" * 40)

# Execute the async loop inside Google Colab
await run_team()

--- Starting Multi-Agent Session ---

[user]: Explain why Groq LPUs provide higher throughput for LLMs than standard GPUs.
----------------------------------------

[Researcher]: **Groq LPUs and their Advantages over Standard GPUs for Large Language Models (LLMs)**

### Introduction

In recent years, there has been an explosive growth in the field of Large Language Models (LLMs) such as BERT, RoBERTa, and T5, among others. These models have achieved state-of-the-art performance in numerous natural language processing (NLP) tasks. However, their massive computational requirements have made it challenging to train and deploy them efficiently. Groq Launch Pad Units (LPUs) have emerged as a promising solution for addressing these challenges. In this section, we'll delve into the reasons why Groq LPUs provide higher throughput for LLMs than standard GPUs.

### Key Characteristics of Groq LPUs

Groq LPUs are purpose-built chips designed specifically for machine learning (ML) workloads, espec

In [ ]:
import os

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.conditions import TextMentionTermination

groq_key = os.environ.get("GROQ_API_KEY", "")

agent = AssistantAgent(
    name="weather_agent",
    system_message="You are a helpful weather assistant.",
    model_client=OpenAIChatCompletionClient(
        model="llama-3.1-8b-instant",
        api_key=groq_key,
        base_url="https://api.groq.com/openai/v1",
        model_info={
            "vision": False,
            "function_calling": True,
            "json_output": True,
            "family": "llama",
            "structured_output": False
        }
    ),
)

agent_team = RoundRobinGroupChat(
    participants=[agent],
    termination_condition=TextMentionTermination("TERMINATE")
)

config = agent_team.dump_component()
print(config.model_dump_json(indent=2))

{
  "provider": "autogen_agentchat.teams.RoundRobinGroupChat",
  "component_type": "team",
  "version": 1,
  "component_version": 1,
  "description": "A team that runs a group chat with participants taking turns in a round-robin fashion\n    to publish a message to all.",
  "label": "RoundRobinGroupChat",
  "config": {
    "participants": [
      {
        "provider": "autogen_agentchat.agents.AssistantAgent",
        "component_type": "agent",
        "version": 1,
        "component_version": 1,
        "description": "An agent that provides assistance with tool use.",
        "label": "AssistantAgent",
        "config": {
          "name": "weather_agent",
          "model_client": {
            "provider": "autogen_ext.models.openai.OpenAIChatCompletionClient",
            "component_type": "model",
            "version": 1,
            "component_version": 1,
            "description": "Chat completion client for OpenAI hosted models.",
            "label": "OpenAIChatCompletionCl

In [ ]:
with open("team.json", "w") as f:
    f.write(config.model_dump_json())

print("Saved config to team.json")

Saved config to team.json


In [ ]:
!autogenstudio serve --team team.json --port 8084

INFO:     Started server process [40623]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8084 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [40623]
